# Forge Academy: Data Science Foundations
## Week 2, Session 2: Working with Real CSV Data (Titanic Passenger Records)

In Session 1 you stored EkoMart orders in lists and dictionaries that you typed yourself. Real data does not arrive that way. It arrives as **files**: exports from a database, downloads from a website, spreadsheets saved as CSV. Today you will download a real public dataset from Kaggle, the passenger list of the RMS Titanic (1912), and treat it the way a working analyst would.

Your brief: a museum in Lagos is preparing a Titanic exhibition and has asked Forge Academy for a **clean, trustworthy passenger file** and a few verified facts for the display boards. The raw file has missing ages, missing cabins, missing ports, zero fares and names in an awkward format. Your job is to read it, inspect it, validate it, clean it and save a cleaned copy, **without ever changing the raw file**.

**Dataset:** Titanic Dataset, published on Kaggle by M Yasser H: https://www.kaggle.com/datasets/yasserh/titanic-dataset (891 passengers, 12 columns).

### The rule for today

> **Inspect before you clean. Save a cleaned copy. Keep the raw file.**

### Learning objectives

By the end of this session you should be able to:

1. Open, read and write plain text files with `open()` and `with`, work with file paths, and handle a missing file.
2. Download a public Kaggle dataset, and fall back to uploading a file by hand.
3. Explain what a CSV file is, why splitting lines on commas breaks, and read and write CSVs with the `csv` module.
4. Inspect a dataset before changing it: size, columns, types, blanks, unique values, ranges and duplicates.
5. Validate rows with conditionals and record exactly which rows fail and why.
6. Clean data step by step: column names, values inside fields, out-of-range values, duplicates, text with regular expressions, accents, near-miss spellings and dates.
7. Build a reusable cleaning pipeline from functions and save `cleaned_titanic.csv` next to the untouched raw file.



## 2. Files, paths and missing files

### 2.1 Plain text files

A **plain text file** (`.txt`, `.csv`, `.py`) stores characters only: no fonts, no formulas, no colours.

| Pros | Cons |
|---|---|
| Opens on any computer and in any program | No built-in structure or types: everything is text |
| Small and easy to share | No formatting or formulas |
| Easy for Python to read and write | Easy to break with one wrong comma or missing line |

`open(path, mode)` opens a file. Modes: `"r"` read (default), `"w"` write (**erases** the file first), `"a"` append. Always use `with`: it closes the file for you, even if an error happens.

In [2]:
with open("../data/museum_notes.txt", "w", encoding="utf-8") as f:
    f.write("Titanic exhibition notes\n")
    f.write("Sailed from Southampton on 10 April 1912\n")

In [ ]:
with open("museum_notes.txt", "a", encoding="utf-8") as f:
    f.write("Stopped at Cherbourg and Queenstown\n")

In [ ]:

with open("museum_notes.txt", encoding="utf-8") as f:
    content = f.read()            # the whole file as one string
print(content)

In [ ]:
print(type(content), len(content), "characters")

In [ ]:
with open("museum_notes.txt", encoding="utf-8") as f:
    first = f.readline()          # one line, including "\n"
    rest = f.readlines()          # the remaining lines as a list
print(repr(first))
print(rest)

with open("museum_notes.txt", encoding="utf-8") as f:
    for number, line in enumerate(f, start=1):   # the memory-friendly way for big files
        print(number, line.strip())

### 2.2 File paths

A **path** says where a file lives. `museum_notes.txt` is a **relative** path (relative to the current folder). `/content/museum_notes.txt` is an **absolute** path. In Colab your current folder is `/content`. Use `os.path.join` to build paths instead of gluing strings together.

In [ ]:
import os
print("Current folder:", os.getcwd())
print("Files here:", sorted(os.listdir("."))[:10])
print(os.path.join("data", "raw", "titanic.csv"))
print(os.path.exists("museum_notes.txt"), os.path.exists("no_such_file.txt"))

### 2.3 Handling a missing file

Opening a file that is not there raises `FileNotFoundError`. Either check first with `os.path.exists`, or catch the error.

In [ ]:
open("passengers_2024.csv")     # read the last line of the error

In [ ]:
def read_text(path):
    try:
        with open(path, encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        print(f"Could not find {path}. Check the name and the folder.")
        return None

print(read_text("passengers_2024.csv"))
print(read_text("museum_notes.txt")[:24])

**Try it 2A:** Write a file `my_goals.txt` with three lines (one goal per line). Read it back and print each line with its number, like `Goal 1: ...`.

In [ ]:
# Try it 2A

## 3. Getting the data from Kaggle

Kaggle (kaggle.com) hosts thousands of free datasets. Our dataset page is https://www.kaggle.com/datasets/yasserh/titanic-dataset. Open it in a new tab now and look at the **Data Card**: it explains each column. Reading the description is part of inspecting.

| Column | Meaning |
|---|---|
| PassengerId | Row identifier |
| Survived | 0 = did not survive, 1 = survived |
| Pclass | Ticket class: 1 = first, 2 = second, 3 = third |
| Name | "Surname, Title. First names" |
| Sex | male or female |
| Age | Age in years; fractions for babies; blank if unknown |
| SibSp | Number of siblings or spouses aboard |
| Parch | Number of parents or children aboard |
| Ticket | Ticket number |
| Fare | Ticket price in British pounds (1912) |
| Cabin | Cabin number; blank if unknown |
| Embarked | Port: S = Southampton, C = Cherbourg, Q = Queenstown |

### Option 1 (use this): `kagglehub`

`kagglehub` is Kaggle's official Python library and comes installed in Colab. For a public dataset it downloads without any login. It saves the file in a cache folder and tells us where.

In [ ]:
import os

RAW_PATH = None
try:
    import kagglehub                      # already installed in Colab
    folder = kagglehub.dataset_download("yasserh/titanic-dataset")
    RAW_PATH = os.path.join(folder, "Titanic-Dataset.csv")
except Exception as error:
    print("kagglehub download did not work:", error)

# Fallback: a file you uploaded through the Colab Files panel
if (RAW_PATH is None or not os.path.exists(RAW_PATH)) and os.path.exists("Titanic-Dataset.csv"):
    RAW_PATH = "Titanic-Dataset.csv"

print("Raw file:", RAW_PATH)
print("Exists:", RAW_PATH is not None and os.path.exists(RAW_PATH))

### Option 2: the Kaggle API with your own key

Use this if Option 1 fails or for datasets that need you to be signed in.

1. On kaggle.com, sign in, go to **Settings > API > Create New Token**. A file `kaggle.json` downloads.
2. Run the cell below. It asks you to upload `kaggle.json`, then downloads and unzips the dataset into `/content`.
3. Never share `kaggle.json` or put it in a public notebook: it is your password.

In [ ]:
from google.colab import files
files.upload()                     # choose kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d yasserh/titanic-dataset --unzip
RAW_PATH = "Titanic-Dataset.csv"

### Option 3: upload by hand

1. On the Kaggle page, click **Download** and unzip the file on your computer.
2. In Colab, click the **folder icon** on the left, then the **upload icon**, and choose `Titanic-Dataset.csv`.
3. Run the Option 1 cell again; its fallback finds the uploaded file.

**Golden rule:** from now on, `RAW_PATH` is read-only. We never open it with `"w"`. Everything we clean goes into a new file.

## 4. What is a CSV file?

CSV means **comma-separated values**. It is a plain text file where each line is a row and commas separate the columns. The first line is usually the **header**. Let us look at the raw text before Python interprets anything.

In [ ]:
with open(RAW_PATH, encoding="utf-8") as f:
    for i in range(4):
        print(repr(f.readline()))

### 4.1 Manual parsing, and why it breaks

It seems natural to split each line on commas. Look at the first passenger's name.

In [ ]:
with open(RAW_PATH, encoding="utf-8") as f:
    header = f.readline().strip().split(",")
    first = f.readline().strip().split(",")

print(len(header), "header fields")
print(len(first), "fields in row 1")
print(first)

**Discuss:** Why does row 1 have 13 pieces when there are 12 columns? Look at `"Braund, Mr. Owen Harris"`.

The name contains a comma, so the file wraps it in quotes. A plain `split(",")` does not understand quotes. The `csv` module does. This is the main reason to use it: it handles quotes, commas inside values and line endings correctly.

### 4.2 `csv.reader`: each row as a list

In [ ]:
import csv

with open(RAW_PATH, newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader)          # the first row
    first = next(reader)

print(len(first), "fields")
print(first)
print(f"{first[3]} | class {first[2]} | fare {first[9]}")

### 4.3 `csv.DictReader`: each row as a dictionary

This is the list-of-dictionaries structure from Session 1, built for you from the header.

In [ ]:
import csv

with open(RAW_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    columns = reader.fieldnames
    rows = list(reader)

print("Columns:", columns)
print("Rows:", len(rows))

In [ ]:
print(rows[0])
print()
for row in rows[:5]:
    print(f"{row['PassengerId']:>3}  {row['Name'][:30]:<30}  class {row['Pclass']}  age {row['Age'] or '?':>4}  fare {row['Fare']}")

### 4.4 Writing a CSV and reading it back

`csv.writer` writes lists; `csv.DictWriter` writes dictionaries. Open with `newline=""` so Windows does not add blank lines. Always read the new file back to check it.

In [ ]:
sample = rows[:3]
with open("titanic_sample.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["PassengerId", "Name", "Survived"], extrasaction="ignore")
    writer.writeheader()
    writer.writerows(sample)

with open("titanic_sample.csv", newline="", encoding="utf-8") as f:
    print(f.read())

**Try it 4A:** Using `csv.writer` (lists, not dictionaries), write a file `ports.csv` with a header `code,port` and three rows for S, C and Q. Read it back with `csv.reader` and print each row with an f-string: `S means Southampton`.

In [ ]:
# Try it 4A

## 5. Inspect before you clean

Cleaning without inspecting is guessing. Answer these questions first, and write down what you find.

1. How many rows and columns are there?
2. What type is each value **really**?
3. Where are the blanks?
4. What values appear in each category column?
5. What are the ranges of the number columns?
6. Are there duplicates?

### 5.1 Every value is text

In [ ]:
first = rows[0]
for column in columns:
    print(f"{column:<12} {first[column]!r:<28} {type(first[column]).__name__}")

`Age` looks like a number but it is the string `'22'`. **Assuming types** is a classic mistake: `"22" + 1` fails, and `"9" > "10"` is `True` because text compares letter by letter.

In [ ]:
print("9" > "10", 9 > 10)
print(sorted(["9", "10", "100", "2"]))

### 5.2 Blanks per column

In [ ]:
blanks = {column: 0 for column in columns}
for row in rows:
    for column in columns:
        if row[column].strip() == "":
            blanks[column] += 1

for column, count in blanks.items():
    if count:
        print(f"{column:<10} {count:>4} blank ({count / len(rows):.1%})")

### 5.3 Category columns: what values appear?

In [ ]:
from collections import Counter

for column in ["Survived", "Pclass", "Sex", "Embarked"]:
    print(column, dict(Counter(row[column] for row in rows)))

### 5.4 Number columns: ranges and conversion problems

In [ ]:
def number_summary(rows, column):
    values, bad = [], 0
    for row in rows:
        try:
            values.append(float(row[column]))
        except ValueError:
            bad += 1
    print(f"{column}: {len(values)} numbers, {bad} not numbers, min {min(values)}, max {max(values)}")
    return values

ages = number_summary(rows, "Age")
fares = number_summary(rows, "Fare")
print("Fractional ages:", sorted(a for a in ages if a != int(a))[:8])
print("Zero fares:", sum(1 for f in fares if f == 0))

### 5.5 Duplicates and identifiers

In [ ]:
ids = [row["PassengerId"] for row in rows]
print("Rows:", len(ids), "| Unique ids:", len(set(ids)))

tickets = Counter(row["Ticket"] for row in rows)
print("Most shared tickets:", tickets.most_common(3))

`PassengerId` is unique, so it is a good **identifier**. Tickets repeat because families shared a ticket, so a repeated ticket is **not** a duplicate row. Knowing the difference is why we inspect.

**Discuss:** Write an inspection summary in three bullet points. What would you fix, what would you leave alone, and what would you ask the data owner?

**Try it 5A:** Count passengers by `Pclass` and, for each class, how many have a blank `Cabin`. Which class is missing cabins most? What might that tell us about how the data was collected?

In [ ]:
# Try it 5A

## 6. Validate, clean and save the Titanic file

### 6.1 The preprocessing workflow

1. **Load** the raw file (read-only).
2. **Inspect** it (section 5).
3. **Fix structure**: column names, row lengths.
4. **Standardise values**: types, spelling, case, codes.
5. **Validate**: rules for each field; record failures with their row number.
6. **Handle faulty records**: fix, mark as missing, or remove, and say why.
7. **Save** a cleaned copy and **read it back**.

### 6.2 Validating with conditionals

A validation rule says what a correct value looks like. We track the **line number** of each failure so we can find it in the file (line 1 is the header, so the first data row is line 2).

In [ ]:
def check_raw_row(row):
    problems = []
    if row["Survived"] not in ("0", "1"):
        problems.append("Survived not 0/1")
    if row["Sex"] not in ("male", "female"):
        problems.append("Sex unexpected")
    if row["Age"] == "":
        problems.append("Age blank")
    if row["Embarked"] == "":
        problems.append("Embarked blank")
    if row["Fare"] in ("0", "0.0"):
        problems.append("Fare is zero")
    return problems

problem_count = Counter()
examples = {}
for line_number, row in enumerate(rows, start=2):
    for p in check_raw_row(row):
        problem_count[p] += 1
        examples.setdefault(p, line_number)

for p, n in problem_count.items():
    print(f"{p:<16} {n:>4} rows  (first at line {examples[p]})")

### 6.3 Deciding what to do with each problem

| Problem | Decision | Why |
|---|---|---|
| Age blank (177) | Keep the row, store age as empty, add `age_missing` | Removing 20% of passengers would bias the exhibition |
| Cabin blank (687) | Keep, extract the deck letter where known, else `Unknown` | Mostly third class: the gap is informative |
| Embarked blank (2) | Keep, store `Unknown` | Too few to matter; do not invent a port |
| Fare zero (15) | Keep, but flag for the museum to check | Could be crew or complimentary tickets |
| Name format | Extract the title (Mr, Mrs, Miss) and normalise rare spellings | Useful for analysis and display |

### 6.4 Cleaning one row at a time

In [ ]:
import csv, os, re

PORTS = {"S": "Southampton", "C": "Cherbourg", "Q": "Queenstown"}
TITLE_FIX = {"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"}

def load_csv(path):
    """Read a CSV file and return (column_names, list_of_row_dicts)."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"No file at {path}")
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        return reader.fieldnames, list(reader)

def to_number(text, kind=float):
    """Convert text to a number; return None if it is blank or invalid."""
    try:
        return kind(text)
    except (ValueError, TypeError):
        return None

def extract_title(name):
    match = re.search(r",\s*([^.]+)\.", name)
    if not match:
        return "Unknown"
    title = match.group(1).strip()
    return TITLE_FIX.get(title, title)

def clean_row(row):
    """Return a NEW cleaned dict. The input row is not changed."""
    cabin = row["Cabin"].strip()
    sibsp = to_number(row["SibSp"], int)
    parch = to_number(row["Parch"], int)
    return {
        "passenger_id": to_number(row["PassengerId"], int),
        "survived": to_number(row["Survived"], int),
        "pclass": to_number(row["Pclass"], int),
        "name": " ".join(row["Name"].split()),
        "title": extract_title(row["Name"]),
        "sex": row["Sex"].strip().lower(),
        "age": to_number(row["Age"]),
        "age_missing": row["Age"].strip() == "",
        "family_size": sibsp + parch + 1,
        "fare": to_number(row["Fare"]),
        "deck": cabin[0] if cabin else "Unknown",
        "embarked": PORTS.get(row["Embarked"].strip(), "Unknown"),
    }

def validate_row(clean):
    """Return a list of problems with a cleaned row (empty list = OK)."""
    problems = []
    if clean["survived"] not in (0, 1):
        problems.append("survived must be 0 or 1")
    if clean["pclass"] not in (1, 2, 3):
        problems.append("pclass must be 1, 2 or 3")
    if clean["sex"] not in ("male", "female"):
        problems.append("unexpected sex value")
    if clean["age"] is not None and not (0 < clean["age"] <= 100):
        problems.append("age out of range")
    if clean["fare"] is None or clean["fare"] < 0:
        problems.append("fare missing or negative")
    return problems

def save_csv(rows, path, raw_path):
    """Write rows to path, refusing to overwrite the raw file."""
    if os.path.abspath(path) == os.path.abspath(raw_path):
        raise ValueError("Refusing to overwrite the raw file")
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

def run_pipeline(raw_path, out_path):
    columns, raw_rows = load_csv(raw_path)
    cleaned, rejected = [], []
    seen_ids = set()
    for line_number, row in enumerate(raw_rows, start=2):   # line 1 is the header
        clean = clean_row(row)
        problems = validate_row(clean)
        if clean["passenger_id"] in seen_ids:
            problems.append("duplicate passenger_id")
        seen_ids.add(clean["passenger_id"])
        if problems:
            rejected.append((line_number, problems))
        else:
            cleaned.append(clean)
    save_csv(cleaned, out_path, raw_path)
    print(f"Read {len(raw_rows)} rows, kept {len(cleaned)}, rejected {len(rejected)}")
    print(f"Saved {out_path}")
    return cleaned, rejected

In [ ]:
print(rows[0])
print()
print(clean_row(rows[0]))
print(validate_row(clean_row(rows[0])))
print("Raw row unchanged:", rows[0]["Age"], type(rows[0]["Age"]).__name__)

**Discuss:** `clean_row` builds a **new** dictionary. Why is that safer than editing `row` in place? (Think back to the list-mutation trap in Session 1.)

### 6.5 Extracting text with a regular expression

`extract_title` uses `re.search(r",\s*([^.]+)\.", name)`, which means: after a comma and any spaces, capture everything up to the first full stop.

In [ ]:
import re
for name in ["Braund, Mr. Owen Harris", "Rothes, the Countess. of (Lucy Noel Martha Dyer-Edwards)", "Aubart, Mme. Leontine Pauline"]:
    print(f"{name[:40]:<40} -> {extract_title(name)}")

print(Counter(extract_title(r["Name"]) for r in rows).most_common(6))

### 6.6 Save the cleaned copy and check it

In [ ]:
raw_size_before = os.path.getsize(RAW_PATH)

cleaned, rejected = run_pipeline(RAW_PATH, "cleaned_titanic.csv")

print("Raw file unchanged:", os.path.getsize(RAW_PATH) == raw_size_before)
check_columns, check_rows = load_csv("cleaned_titanic.csv")
print("Cleaned columns:", check_columns)
print("Cleaned rows:", len(check_rows))
print(check_rows[0])

All 891 rows pass validation: this file's problems are **missing** values, which we keep and mark, not **invalid** ones. The rejection list matters on messier data, as section 7 shows.

Notice that the read-back values are text again (`'22.0'`, `'True'`). A CSV stores only text; types are restored each time you load. Also, a missing age is saved as an empty field, which is exactly how the raw file represented it.

In [ ]:
save_csv(cleaned, RAW_PATH, RAW_PATH)     # the guard in save_csv protects the raw file

## 7. A messier file: Lagos ferry bookings

The Titanic file is fairly well structured. Real exports from forms are worse. The cell below creates a small file of Lagos ferry bookings with the problems from EduBEE Module 3 built in. Inspect it first.

In [ ]:
ferry_text = """Booking ID, Passenger Name ,AGE ,Gender,Route,Travel Date,Phone,Fare (NGN)
B001,Ada Okafor,29,Female,Ikorodu-CMS,2026-10-03,08031234567,1500
B002, tunde  bakare ,41,M,ikorodu-cms,03/10/2026,+234 802 555 1234,1500
B003,Chioma Eze,twenty,F,Ikoyi-Ikorodu,2026-10-04,0803-111-2222,2000
B004,Emeka Obi,-5,male,Ikorod-CMS,04/10/2026,08099998888,1500
B005,Zainab Musa,150,female,Ìkòyí-Ikorodu,2026/10/05,0701 234 5678,2000
B002, tunde  bakare ,41,M,ikorodu-cms,03/10/2026,+234 802 555 1234,1500
B006,Bola Ade,35,Male,Badore-Ijede,05-10-2026,08123456789
B007,Kemi Lawal,27,FEMALE,Badore-Ijede,2026-10-06,08134567890,N/A
"""
with open("ferry_bookings_raw.csv", "w", encoding="utf-8") as f:
    f.write(ferry_text)
print("Saved ferry_bookings_raw.csv")

In [ ]:
with open("ferry_bookings_raw.csv", newline="", encoding="utf-8") as f:
    ferry_rows = list(csv.reader(f))

ferry_header = ferry_rows[0]
print(ferry_header)
for index, row in enumerate(ferry_rows[1:], start=1):
    print(index, len(row), row)

**Discuss:** List every problem you can see before reading on. There are at least eight kinds.

### 7.1 Structural errors and column names

In [ ]:
def standardise_column(name):
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)     # anything not a letter or digit becomes _
    return name.strip("_")

columns_clean = [standardise_column(c) for c in ferry_header]
print(columns_clean)

good_rows, broken = [], []
for line_number, row in enumerate(ferry_rows[1:], start=2):
    if len(row) != len(columns_clean):
        broken.append((line_number, row))
    else:
        good_rows.append(dict(zip(columns_clean, row)))
print("Structurally broken:", broken)
print(good_rows[0].keys())

### 7.2 Removing duplicates with a set, and normalising before comparing

In [ ]:
def normalise_text(text):
    return " ".join(text.split()).lower()

seen = set()
unique_rows = []
for row in good_rows:
    key = (row["booking_id"], normalise_text(row["passenger_name"]))
    if key in seen:                    # set membership is fast
        print("Duplicate removed:", row["booking_id"])
        continue
    seen.add(key)
    unique_rows.append(row)
print(len(good_rows), "->", len(unique_rows))

### 7.3 Standardising values inside fields

In [ ]:
GENDER_MAP = {"m": "Male", "male": "Male", "f": "Female", "female": "Female"}

for row in unique_rows:
    row["passenger_name"] = " ".join(row["passenger_name"].split()).title()
    row["gender"] = GENDER_MAP.get(row["gender"].strip().lower(), "Unknown")
    print(f"{row['booking_id']}  {row['passenger_name']:<14} {row['gender']}")

### 7.4 Out-of-range and invalid values

In [ ]:
for row in unique_rows:
    age = to_number(row["age"], int)
    if age is None:
        print(f"{row['booking_id']}: age {row['age']!r} is not a number -> set to None")
    elif not 0 <= age <= 110:
        print(f"{row['booking_id']}: age {age} is out of range -> set to None")
        age = None
    row["age"] = age

### 7.5 Unicode and fuzzy matching

`Ìkòyí` has accents and `Ikorod` is a typo. `unicodedata` removes accents; `difflib.get_close_matches` finds the nearest valid name.

In [ ]:
import unicodedata, difflib

VALID_ROUTES = ["Ikorodu-CMS", "Ikoyi-Ikorodu", "Badore-Ijede"]

def strip_accents(text):
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")

def fix_route(route):
    plain = strip_accents(route).strip().lower()
    lookup = {r.lower(): r for r in VALID_ROUTES}
    match = difflib.get_close_matches(plain, lookup.keys(), n=1, cutoff=0.8)
    return lookup[match[0]] if match else "Unknown"

for row in unique_rows:
    print(f"{row['route']!r:<18} -> {fix_route(row['route'])}")
    row["route"] = fix_route(row["route"])

### 7.6 Regex for phone numbers, and chained string operations

`re.sub(r"\D", "", text)` deletes every non-digit. Then we convert `234...` to the local `0...` format.

In [ ]:
def clean_phone(text):
    digits = re.sub(r"\D", "", text)
    if digits.startswith("234"):
        digits = "0" + digits[3:]
    return digits if len(digits) == 11 else None

for row in unique_rows:
    print(f"{row['phone']!r:<20} -> {clean_phone(row['phone'])}")
    row["phone"] = clean_phone(row["phone"])

### 7.7 Date strings

Dates arrive in many formats. `datetime.strptime` reads a string using a format you give it; we try each known format in turn.

In [ ]:
from datetime import datetime

DATE_FORMATS = ["%Y-%m-%d", "%d/%m/%Y", "%Y/%m/%d", "%d-%m-%Y"]

def parse_date(text):
    for fmt in DATE_FORMATS:
        try:
            return datetime.strptime(text.strip(), fmt).date()
        except ValueError:
            continue
    return None

for row in unique_rows:
    parsed = parse_date(row["travel_date"])
    print(f"{row['travel_date']:<12} -> {parsed}  ({parsed.strftime('%A') if parsed else 'invalid'})")
    row["travel_date"] = parsed.isoformat() if parsed else ""

### 7.8 Resetting ids and saving

In [ ]:
for new_id, row in enumerate(unique_rows, start=1):
    row["row_id"] = new_id
    row["fare_ngn"] = to_number(row["fare_ngn"], int)

with open("cleaned_ferry_bookings.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["row_id"] + columns_clean)
    writer.writeheader()
    writer.writerows(unique_rows)

with open("cleaned_ferry_bookings.csv", encoding="utf-8") as f:
    print(f.read())

**Discuss:** B006 was removed because its row was broken. Was that the right decision? What else could you do? (Hint: which field is probably missing?)

## 8. Debugging clinic

Each of the next six cells has one intentional error. Run it, read the last line of the error, fix it, and explain the fix in one sentence. Cell 6 does not crash, but it does something dangerous.

In [ ]:
with open("Titanic-dataset.csv") as f:
    print(f.readline())

In [ ]:
total_age = 0
for row in rows:
    total_age = total_age + row["Age"]

In [ ]:
with open(RAW_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
print(next(reader))

In [ ]:
fares = [float(row["fare"]) for row in rows]

In [ ]:
ages = [float(row["Age"]) for row in rows]

In [ ]:
# Cell 6: do NOT fix by running it on RAW_PATH. Explain what would happen if you did.
def save_clean(rows, path):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

save_clean(cleaned, "cleaned_titanic_v2.csv")     # safe here; imagine path=RAW_PATH

## 9. Guided practice: use your pipeline

Section 6.4 gave you a pipeline: `load_csv`, `to_number`, `extract_title`, `clean_row`, `validate_row`, `save_csv` and `run_pipeline`. The tasks below use `cleaned` (the list of cleaned dictionaries). Use functions where the task asks for one.

### Task A: Survival rate by class

Write `survival_rate(rows)` that returns the percentage of passengers in `rows` who survived. Use it to print the rate for each class (1, 2, 3).

In [ ]:
# Task A

### Task B: Survival by sex and port

Print the survival rate for `male` and `female`, then for each `embarked` port. Reuse `survival_rate`.

In [ ]:
# Task B

### Task C: Average age by title

For the titles Mr, Mrs, Miss and Master, print how many passengers have a known age and their average age. Skip missing ages (`None`). What does "Master" mean in 1912 English?

In [ ]:
# Task C

### Task D: Travelling alone

`family_size` is 1 for someone travelling alone. Compare the survival rate of people travelling alone with people travelling with family.

In [ ]:
# Task D

### Task E: Exhibition file

Create `exhibition_first_class.csv` with only first-class passengers and the columns `name`, `title`, `age`, `deck`, `survived`, sorted by name. Read it back and print the row count and the first three rows. Make sure you did not touch `RAW_PATH`.

In [ ]:
# Task E

## 10. Challenge: fill missing ages responsibly

The museum wants an age shown for every passenger, clearly marked as estimated where needed.

1. Calculate the **median** known age for each title (use `statistics.median`).
2. Create a new list where each passenger with a missing age gets the median for their title, and add a column `age_estimated` (`True` or `False`).
3. Save it as `cleaned_titanic_estimated_ages.csv`. Keep `cleaned_titanic.csv` as it is.
4. Print how many ages were estimated and the median used for each title.
5. In a markdown cell, explain one risk of filling missing values, and why the `age_estimated` column matters.

In [ ]:
# Challenge

## 11. Preview: the same work in pandas (Week 3)

Everything you did today by hand is what pandas does for you. You now know what is happening underneath.

In [ ]:
import pandas as pd

df = pd.read_csv(RAW_PATH)
print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df.groupby("Pclass")["Survived"].mean().round(3))

## 12. Recap

| Step | Question | Python tool |
|---|---|---|
| Open | Where is the file, and does it exist? | `open`, `with`, `os.path.exists`, `FileNotFoundError` |
| Read | Rows and columns | `csv.reader`, `csv.DictReader` (not `split(",")`) |
| Inspect | Size, types, blanks, categories, ranges, duplicates | loops, `Counter`, `set`, `try/except float()` |
| Validate | Which rows break which rule? | conditionals, `enumerate` for line numbers |
| Clean | Structure, names, values, ranges, text, dates | `str` methods, `re`, `unicodedata`, `difflib`, `datetime` |
| Save | Write a new file and read it back | `csv.DictWriter`, `newline=""` |
| Protect | Raw file never changes | separate output names, a guard in `save_csv` |

Three mistakes to avoid: cleaning before inspecting, assuming types (CSV values are always text), and overwriting the raw file.

**Exit question:** Why does `"Braund, Mr. Owen Harris"` break `split(",")` but not `csv.reader`?

## Week 2 project

**Due:** before Week 3, Session 1. Submit the link to your Colab notebook through the course form, with sharing set to "Anyone with the link can view". Also complete EduBEE PD101 Module 2 and Module 3 (I/O and CSV) if you have not yet.

Choose **one** of the four projects below. Build it in a new Colab notebook (or below this cell). Every project must:

- use at least **three functions** that return values;
- use at least **one list, one dictionary and one set or tuple**;
- read from **and** write to a CSV file using the `csv` module;
- handle at least **one error** with a specific `try/except`;
- start with a markdown cell explaining what the program does and end with a short reflection (what was hard, what you would improve).

### Option 1: Student Records Manager

Keep a file `students.csv` with `student_id, name, track, score`. Write functions to add a student, find a student by id, update a score, delete a student, list students scoring below a pass mark, and print class statistics (average, highest, lowest). Save after each change. Reject duplicate ids and scores outside 0 to 100.

### Option 2: Expense Tracker

Keep `expenses.csv` with `date, category, description, amount`. Write functions to add an expense (validate the date with `datetime` and the amount as a positive number), total spending by category, total by month, and show the biggest expense. Export a summary file `expense_summary.csv`.

### Option 3: Quiz

Store at least 10 questions in `questions.csv` with `question, option_a, option_b, option_c, answer, topic`. Load them, ask them with `input()`, accept answers in any case (`"a"` or `"A"`), reject invalid choices without crashing, score the player, show the score per topic, and append the result to `results.csv` with the player's name and the date.

### Option 4: CSV Data Cleaner

Pick a different small, messy public dataset from Kaggle (under 5,000 rows; check with Adam if unsure). Write an inspection report (rows, columns, blanks, categories, ranges, duplicates), clean it with a pipeline of functions, save `cleaned_<name>.csv` without changing the raw file, and write five verified facts from the cleaned data. Include the Kaggle URL.

### How projects are marked (out of 20)

| Criterion | 4 points | 2 points | 0 points |
|---|---|---|---|
| Works | Runs top to bottom without errors | Runs with small errors | Does not run |
| Structure | 3+ clear functions with `return`; good names | Functions present but mostly print | No functions |
| Data structures | Appropriate list, dict and set/tuple use | Some used, some forced | Only plain variables |
| Files and validation | Reads and writes CSV; validates input; handles an error; raw file safe | Partly | Missing |
| Explanation | Clear intro, comments where needed, honest reflection | Brief | None |

Also complete **Task F** below if you did not finish it in class: Tasks A to E from section 9.

**Task F (written response):** Answer in plain language.

1. Name three problems you found when inspecting the Titanic file, and what you decided to do about each.
2. Why did we keep passengers with missing ages instead of removing them?
3. Give one reason never to overwrite a raw data file.

*Your answers:*